In [0]:
%run ./01-config

In [0]:
class SetHhelper():
    def __init__(self, env, db_name):
        Conf = Config()
        self.landing_zone = Conf.base_dir_data + "/raw"
        self.checkpoint_base = Conf.base_dir_data + "/checkpoints"
        self.catalog = env
        self.db_name = db_name
        self.initialized = False
        self.catalog_check = False
    
    def create_catalog(self):
        print(f"Creating the catalog {self.catalog}...", end = '')
        spark.sql(f"CREATE CATALOG IF NOT EXISTS {self.catalog}")
        self.catalog_check = True
        print("Done")

    def create_db(self):
        if(self.catalog_check):
            print(f"Creating the database {self.catalog}.{self.db_name}...", end = '')
            spark.sql(f"CREATE DATABASE IF NOT EXISTS {self.catalog}.{self.db_name}")
            spark.sql(f"USE {self.catalog}.{self.db_name}")
            self.initialized = True
            print("Done")
        else:
            raise ReferenceError("Catalog is not defined. Cannot create database.")

    #Below lines of codes is for creating tables for CRM source
    def create_crm_cust_info(self):
        if (self.initialized):
            print(f"Creating the database {self.catalog}.{self.db_name}.crm_cust_info...", end = '')
            spark.sql(f"""CREATE OR REPLACE Table {self.catalog}.{self.db_name}.crm_cust_info (
                            cst_id INT,
                            cst_key VARCHAR(50),
                            cst_firstname VARCHAR(50),
                            cst_lastname VARCHAR(50),
                            cst_marital_status VARCHAR(50),
                            cst_gndr VARCHAR(50),
                            cst_create_date DATE
                        )""")
            print("Done")
        else:
            raise ReferenceError("Application database is not defined. Cannot create table in default database.")

    def create_crm_prd_info(self):
        if (self.initialized):
            print(f"Creating the database {self.catalog}.{self.db_name}.crm_prd_info...", end = '')
            spark.sql(f"""CREATE OR REPLACE Table {self.catalog}.{self.db_name}.crm_prd_info (
                            prd_id	INT,
                            prd_key	VARCHAR(50),
                            prd_name VARCHAR(50),
                            prd_cost INT,
                            prd_line VARCHAR(50),
                            prd_start_dt DATE,
                            prd_end_dt DATE
                        )""")
            print("Done")
        else:
            raise ReferenceError("Application database is not defined. Cannot create table in default database.")

    def create_crm_sales_details(self):
        if (self.initialized):
            print(f"Creating the database {self.catalog}.{self.db_name}.crm_sales_details...", end = '')
            spark.sql(f"""CREATE OR REPLACE Table {self.catalog}.{self.db_name}.crm_sales_details (
                            sls_ord_num	VARCHAR(50),
                            sls_prd_key	VARCHAR(50),
                            sls_cust_id	INT,
                            sls_order_dt INT,
                            sls_ship_dt	INT,
                            sls_due_dt	INT,
                            sls_sales INT,
                            sls_quantity INT,
                            sls_price INT
                        )""")
            print("Done")
        else:
            raise ReferenceError("Application database is not defined. Cannot create table in default database.")

    #Below lines of codes is for creating tables for ERP source
    def create_erp_cust_az12(self):
        if (self.initialized):
            print(f"Creating the database {self.catalog}.{self.db_name}.erp_cust_az12...", end = '')
            spark.sql(f"""CREATE OR REPLACE Table {self.catalog}.{self.db_name}.erp_cust_az12 (
                            cid VARCHAR(50),
                            bdate DATE,
                            gen VARCHAR(50)
                        )""")
            print("Done")
        else:
            raise ReferenceError("Application database is not defined. Cannot create table in default database.")

    def create_erp_loc_a101(self):
        if (self.initialized):
            print(f"Creating the database {self.catalog}.{self.db_name}.erp_loc_a101...", end = '')
            spark.sql(f"""CREATE OR REPLACE Table {self.catalog}.{self.db_name}.erp_loc_a101 (
                            cid	VARCHAR(50),
	                        cntry VARCHAR(50)
                        )""")
            print("Done")
        else:
            raise ReferenceError("Application database is not defined. Cannot create table in default database.")

    def create_erp_px_cat_glv2(self):
        if (self.initialized):
            print(f"Creating the database {self.catalog}.{self.db_name}.erp_px_cat_glv2...", end = '')
            spark.sql(f"""CREATE OR REPLACE Table {self.catalog}.{self.db_name}.erp_px_cat_glv2 (
                            id VARCHAR(50),
                            cat VARCHAR(50),
                            subcat VARCHAR(50),
                            maintenance VARCHAR(50)
                        )""")
            print("Done")
        else:
            raise ReferenceError("Application database is not defined. Cannot create table in default database.")

    def setup(self):
        import time
        start = int(time.time())
        print("\nStarting setup...")
        self.create_catalog()
        self.create_db()
        self.create_crm_cust_info()
        self.create_crm_prd_info()
        self.create_crm_sales_details()
        self.create_erp_cust_az12()
        self.create_erp_loc_a101()
        self.create_erp_px_cat_glv2()
        end = int(time.time())
        print(f"\nSetup completed in {end-start} seconds.")

    def assert_table(self, table_name):
        assert spark.sql(f"SHOW TABLES IN {self.catalog}.{self.db_name}")\
                    .filter(f"isTemporary == False and tableName == '{table_name}'")\
                    .count() == 1, f"Table {table_name} is not found"
        print(f"Found {table_name} table in {self.catalog}.{self.db_name}: Success")

    def validate(self):
        import time
        start = int(time.time())
        print("\nStarting setup validation...")
        self.assert_table("crm_cust_info")
        self.assert_table("crm_prd_info")
        self.assert_table("crm_sales_details")
        self.assert_table("erp_cust_az12")
        self.assert_table("erp_loc_a101")
        self.assert_table("erp_px_cat_glv2")
        end = int(time.time())
        print(f"\nSetup validation completed in {end-start} seconds.")

    def cleanup(self):
        if spark.sql(f"SHOW DATABASES IN {self.catalog}").filter(f"databaseName = '{self.db_name}'").count() == 1:
            print(f"\nDropping the database {self.catalog}.{self.db_name}...", end = '')
            spark.sql(f"DROP DATABASE {self.catalog}.{self.db_name} CASCADE")
            print("Done")
        print(f"\nDeleting {self.landing_zone}...", end = '')
        dbutils.fs.rm(self.landing_zone, True)
        print("Done")
        print(f"\nDeleting {self.checkpoint_base}...", end = '')
        dbutils.fs.rm(self.checkpoint_base, True)
        print("Done")

    